In [9]:
import pandas as pd

from procyclingstats import Race, Stage

import gspread
from google.oauth2.service_account import Credentials
from gspread_dataframe import set_with_dataframe

### Information to get
From api:
- jerseys standings with rider teams and ages
- stage top 5 with rider teams and ages
- combatative rider
- best teammate

From sheet:
- starting lineups

In [10]:
sheet_id = "1CovKWfW7MXokDt-AY6fLI7a5KDtwC781Tj2QbVD7N7o"
gid = "1366607586"
url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv&gid={gid}"
df_lineup = pd.read_csv(url)
for i in range(1, 22):
    df_lineup[str(i)] = df_lineup[str(i)].str.lower()

df_lineup.tail(6)

,user,position,1,2,3,4,5,6,7,8,...,12,13,14,15,16,17,18,19,20,21
39,Paeyton,Team,ef education - easypost,ef education - easypost,ef education - easypost,ef education - easypost,ef education - easypost,ef education - easypost,ef education - easypost,ef education - easypost,...,ef education - easypost,ef education - easypost,ef education - easypost,ef education - easypost,ef education - easypost,ef education - easypost,ef education - easypost,ef education - easypost,ef education - easypost,ef education - easypost
40,Paeyton,Bench,de pooter dries,de pooter dries,de pooter dries,de pooter dries,de pooter dries,de pooter dries,de pooter dries,de pooter dries,...,de pooter dries,de pooter dries,de pooter dries,de pooter dries,de pooter dries,de pooter dries,de pooter dries,de pooter dries,de pooter dries,de pooter dries
41,Paeyton,Bench,lópez harold martín,lópez harold martín,lópez harold martín,lópez harold martín,lópez harold martín,lópez harold martín,lópez harold martín,lópez harold martín,...,lópez harold martín,lópez harold martín,lópez harold martín,lópez harold martín,lópez harold martín,lópez harold martín,lópez harold martín,lópez harold martín,lópez harold martín,lópez harold martín
42,Paeyton,Bench,biermans jenthe,biermans jenthe,biermans jenthe,biermans jenthe,biermans jenthe,biermans jenthe,biermans jenthe,biermans jenthe,...,biermans jenthe,biermans jenthe,biermans jenthe,biermans jenthe,biermans jenthe,biermans jenthe,biermans jenthe,biermans jenthe,biermans jenthe,biermans jenthe
43,Paeyton,Bench,armirail bruno,armirail bruno,armirail bruno,armirail bruno,armirail bruno,armirail bruno,armirail bruno,armirail bruno,...,armirail bruno,armirail bruno,armirail bruno,armirail bruno,armirail bruno,armirail bruno,armirail bruno,armirail bruno,armirail bruno,armirail bruno
44,Paeyton,Bench,riccitello matthew,riccitello matthew,riccitello matthew,riccitello matthew,riccitello matthew,riccitello matthew,riccitello matthew,riccitello matthew,...,riccitello matthew,riccitello matthew,riccitello matthew,riccitello matthew,riccitello matthew,riccitello matthew,riccitello matthew,riccitello matthew,riccitello matthew,riccitello matthew


In [11]:
def load_big_board():
    sheet_id = "1CovKWfW7MXokDt-AY6fLI7a5KDtwC781Tj2QbVD7N7o"
    gid = "360635960"
    url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv&gid={gid}"
    df_big_board = pd.read_csv(url)

    df_big_board['rider_name'] = df_big_board['rider_name'].apply(lambda x: x.lower())
    df_big_board['team_name'] = df_big_board['team_name'].str.replace(r"\s*\(.*?\)", "", regex=True).str.lower()

    return df_big_board

load_big_board().tail(11)

,rider_name,picked,first_name,last_name,nationality,flag,rider_number,team_name,pcs_points_25,pcs_points_24,birthdate,age,age_mult,kom_favorite,sprinter_favorite,young_favorite,overall_favorite
174,bennett george,NaN,George,Bennett,NZ,🇳🇿,221,israel - premier tech,105,339.0,1990-04-07,35,1.0,NaN,NaN,NaN,NaN
175,herrada jesús,✅,JesusHerrada,Lopez,ES,🇪🇸,176,cofidis,197,98.0,1990-07-26,35,1.0,NaN,NaN,NaN,NaN
176,chaves esteban,NaN,JohanEsteban,Chaves,CO,🇨🇴,82,ef education - easypost,147,151.0,1990-01-17,35,1.0,NaN,NaN,NaN,NaN
177,kwiatkowski michał,NaN,Michal,Kwiatkowski,PL,🇵🇱,62,ineos grenadiers,116,157.0,1990-06-02,35,1.0,NaN,NaN,NaN,NaN
178,molard rudy,NaN,Rudy,Molard,FR,🇫🇷,136,groupama - fdj,71,289.0,1989-09-17,35,1.0,NaN,NaN,NaN,NaN
179,juul-jensen christopher,NaN,ChristopherJuul,Jensen,DK,🇩🇰,157,team jayco alula,42,60.0,1989-07-06,36,1.0,NaN,NaN,NaN,NaN
180,de la cruz david,NaN,DavidDeLa,Cruz,ES,🇪🇸,112,q36.5 pro cycling team,318,168.0,1989-05-06,36,1.0,NaN,NaN,NaN,NaN
181,viviani elia,NaN,Elia,Viviani,IT,🇮🇹,207,lotto,91,134.0,1989-02-07,36,1.0,NaN,NaN,NaN,NaN
182,caruso damiano,NaN,Damiano,Caruso,IT,🇮🇹,104,bahrain - victorious,402,144.0,1987-10-12,37,1.0,NaN,NaN,NaN,NaN
183,poels wout,NaN,Wout,Poels,NL,🇳🇱,127,xds astana team,304,448.0,1987-10-01,37,1.0,NaN,NaN,NaN,NaN


In [12]:
def load_team_board():
    sheet_id = "1CovKWfW7MXokDt-AY6fLI7a5KDtwC781Tj2QbVD7N7o"
    gid = "417478782"
    url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv&gid={gid}"
    df = pd.read_csv(url)

    df['team_name'] = df['team_name'].str.replace(r"\s*\(.*?\)", "", regex=True).str.lower()

    return df

load_team_board().head(5)

,rank,prev_rank,team_name,team_url,nationality,flag,class,points,mult
0,1,1,uae team emirates - xrg,team/uae-team-emirates-xrg-2025,AE,🇦🇪,WT,13170,0.1
1,2,2,team visma | lease a bike,team/team-visma-lease-a-bike-2025,NL,🇳🇱,WT,8029,0.1
2,3,3,lidl - trek,team/lidl-trek-2025,US,🇺🇸,WT,8019,0.1
3,4,5,red bull - bora - hansgrohe,team/red-bull-bora-hansgrohe-2025,DE,🇩🇪,WT,7073,0.1
4,5,4,soudal quick-step,team/soudal-quick-step-2025,BE,🇧🇪,WT,6633,0.1


In [13]:
def init_dfs(users: list[str]) -> dict:
    n_stages = 21
    dfs = {}

    for user in users:
        df_user = pd.DataFrame({
            "rider": df_lineup.loc[df_lineup['user'] == user, '1']
        }).reset_index(drop=True)

        for i in range(1, n_stages + 1):
            df_user[f"stage_{i}"] = 0

        dfs[user] = df_user

    return dfs

In [ ]:
race = Race('race/vuelta-a-espana/2025')
winners = race.stages_winners()

completed_stages = []
for i in range(21):
    if winners[i]['rider_name']:
        completed_stages.append(i+1)
print(completed_stages)

Race(url='https://www.procyclingstats.com/race/vuelta-a-espana/2025')
[]


IndexError: list index out of range

In [ ]:
def create_results_dict(stage):
    results = stage.results()
    gc_results = stage.gc()
    points_results = stage.points()
    youth_results = stage.youth()
    kom_results = stage.kom()

    dict = {'stage_results': [r['rider_name'].lower() for r in results if r['rider_name']],
            'gc_leaders': [r['rider_name'].lower() for r in gc_results if r['rider_name']],
            'points_leaders':[r['rider_name'].lower() for r in points_results if r['rider_name']],
            'youth_leaders': [r['rider_name'].lower() for r in youth_results if r['rider_name']],
            'kom_leaders': [r['rider_name'].lower() for r in kom_results if r['rider_name']],
            'stage_bonus': [10, 0, 0, 0, 00, 0, 0, 0, 0, 0, 0, 0, 0, 20, 0, 0, 0, 10, 0, 0, 20]
            }

    return dict

In [ ]:
def create_ttt_results_dict(stage):
    results = stage.results("team_name")
    gc_results = stage.gc()
    points_results = stage.points()
    youth_results = stage.youth()
    kom_results = stage.kom()

    results_dict = {'stage_results': list(dict.fromkeys(r["team_name"].lower() for r in results)),
                    'gc_leaders': [r['rider_name'].lower() for r in gc_results if r['rider_name']],
                    'points_leaders':[r['rider_name'].lower() for r in points_results if r['rider_name']],
                    'youth_leaders': [r['rider_name'].lower() for r in youth_results if r['rider_name']],
                    'kom_leaders': [r['rider_name'].lower() for r in kom_results if r['rider_name']],
                    'stage_bonus': [10, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 20, 0, 0, 0, 10, 0, 0, 20]
                    }

    return results_dict

In [ ]:
def gc_points(stage_number, user, df, results):
    gc_riders = set(df_lineup.loc[(df_lineup['user'] == user) & (df_lineup['position'] == 'GC'), str(stage_number)])
    stage_points = [50, 10, 8, 6, 4]
    stage_bonus = results['stage_bonus'][stage_number-1]

    for i, points in enumerate(stage_points):
        rider = results['stage_results'][i]
        if rider in gc_riders:
            df.loc[df['rider'] == rider, f"stage_{stage_number}"] += points
            if i == 0:
                df.loc[df['rider'] == rider, f"stage_{stage_number}"] += stage_bonus

    leader_categories = ['gc_leaders', 'points_leaders']
    for cat in leader_categories:
        rider = results[cat][0]
        if rider in gc_riders:
            df.loc[df['rider'] == rider, f"stage_{stage_number}"] += 10

In [ ]:
def sprinter_points(stage_number, user, df, results):
    sprinters = set(df_lineup.loc[(df_lineup['user'] == user) & (df_lineup['position'] == 'Sprinter'), str(stage_number)])
    stage_points = [50, 10, 8, 6, 4]
    stage_bonus = results['stage_bonus'][stage_number-1]

    for i, points in enumerate(stage_points):
        rider = results['stage_results'][i]
        if rider in sprinters:
            df.loc[df['rider'] == rider, f"stage_{stage_number}"] += points
            if i == 0:
                df.loc[df['rider'] == rider, f"stage_{stage_number}"] += stage_bonus

    leader_categories = ['gc_leaders', 'points_leaders', 'youth_leaders', 'kom_leaders']
    for cat in leader_categories:
        rider = results[cat][0]
        if rider in sprinters:
            df.loc[df['rider'] == rider, f"stage_{stage_number}"] += 10

In [ ]:
def youth_points(stage_number, user, df, results):
    df_big_board = load_big_board()
    young_riders = set(df_lineup.loc[(df_lineup['user'] == user) & (df_lineup['position'] == 'Young'), str(stage_number)])
    stage_points = [50, 10, 8, 6, 4]
    stage_bonus = results['stage_bonus'][stage_number-1]

    for rider in young_riders:
        age = int(df_big_board.loc[df_big_board['rider_name'] == rider, 'age'].iloc[0])
        age_mult = 0
        if age < 21:
            age_mult = 3
        elif age < 23:
            age_mult = 2
        elif age < 25:
            age_mult = 1.5
        else:
            age_mult = 1

        for i, points in enumerate(stage_points):
            rider = results['stage_results'][i]
            if rider in young_riders:
                df.loc[df['rider'] == rider, f"stage_{stage_number}"] += (age_mult * points)
                if i == 0:
                    df.loc[df['rider'] == rider, f"stage_{stage_number}"] += (age_mult * stage_bonus)

        leader_categories = ['gc_leaders', 'points_leaders', 'youth_leaders', 'kom_leaders']
        for cat in leader_categories:
            rider = results[cat][0]
            if rider in young_riders:
                df.loc[df['rider'] == rider, f"stage_{stage_number}"] += (age_mult * 10)

In [ ]:
def climber_points(stage_number, user, df, results):
    climbers = set(df_lineup.loc[(df_lineup['user'] == user) & (df_lineup['position'] == 'Climber'), str(stage_number)])
    stage_points = [50, 10, 8, 6, 4]
    stage_bonus = results['stage_bonus'][stage_number-1]

    for i, points in enumerate(stage_points):
        rider = results['stage_results'][i]
        if rider in climbers:
            df.loc[df['rider'] == rider, f"stage_{stage_number}"] += points
            if i == 0:
                df.loc[df['rider'] == rider, f"stage_{stage_number}"] += stage_bonus

    leader_categories = ['kom_leaders']
    for cat in leader_categories:
        rider = results[cat][0]
        if rider in climbers:
            df.loc[df['rider'] == rider, f"stage_{stage_number}"] += 10

In [ ]:
def flex_points(stage_number, user, df, results):
    flex_riders = set(df_lineup.loc[(df_lineup['user'] == user) & (df_lineup['position'] == 'Flex'), str(stage_number)])
    stage_points = [50, 10, 8, 6, 4]
    stage_bonus = results['stage_bonus'][stage_number-1]

    for i, points in enumerate(stage_points):
        rider = results['stage_results'][i]
        if rider in flex_riders:
            df.loc[df['rider'] == rider, f"stage_{stage_number}"] += points
            if i == 0:
                df.loc[df['rider'] == rider, f"stage_{stage_number}"] += stage_bonus

    leader_categories = ['gc_leaders', 'points_leaders', 'youth_leaders', 'kom_leaders']
    for cat in leader_categories:
        rider = results[cat][0]
        if rider in flex_riders:
            df.loc[df['rider'] == rider, f"stage_{stage_number}"] += 10

In [ ]:
def team_points(stage_number, user, df, results):
    df_big_board = load_big_board()
    df_teams = load_team_board()
    team = df_lineup.loc[(df_lineup['user'] == user) & (df_lineup['position'] == 'Team'), str(stage_number)].iloc[0]
    riders = set(df_big_board.loc[df_big_board['team_name'] == team, "rider_name"])
    team_mult = df_teams.loc[(df_teams['team_name'] == team), "mult"]

    stage_bonus = results['stage_bonus'][stage_number-1]
    winner = results['stage_results'][0]
    if winner in riders:
        df.loc[df['rider'] == winner, f"stage_{stage_number}"] += (team_mult * 50)
        if i == 0:
            df.loc[df['rider'] == winner, f"stage_{stage_number}"] += (team_mult * stage_bonus)

    leader_categories = ['gc_leaders', 'points_leaders', 'youth_leaders', 'kom_leaders']
    for cat in leader_categories:
        rider = results[cat][0]
        if rider in team:
            df.loc[df['rider'] == rider, f"stage_{stage_number}"] += (team_mult * 10)

In [ ]:
def itt_team_points(stage_number, user, df, results):
    df_teams = load_team_board()
    team = df_lineup.loc[(df_lineup['user'] == user) & (df_lineup['position'] == 'Team'), str(stage_number)].iloc[0]
    team_mult = df_teams.loc[(df_teams['team_name'] == team), "mult"]

    stage_points = [50, 10, 8, 6, 4]
    stage_bonus = results['stage_bonus'][stage_number-1]

    for i, points in enumerate(stage_points):
        winning_teams = results['stage_results'][i]
    if team in winning_teams:
        df.loc[df['rider'] == team, f"stage_{stage_number}"] += (team_mult * points)
        if i == 0:
            df.loc[df['rider'] == team, f"stage_{stage_number}"] += (team_mult * stage_bonus)

    leader_categories = ['gc_leaders', 'points_leaders', 'youth_leaders', 'kom_leaders']
    for cat in leader_categories:
        rider = results[cat][0]
        if rider in team:
            df.loc[df['rider'] == rider, f"stage_{stage_number}"] += (team_mult * 10)

In [ ]:
def user_score(user, df):
    for stage_number in completed_stages:
        print(f"Processing stage {stage_number} for {user}")
        stage = Stage(f"race/vuelta-a-espana/2025/stage-{stage_number}")
        if stage.stage_type() == 'TTT':
            results = create_ttt_results_dict(stage)
            gc_points(stage_number, user, df, results)
            climber_points(stage_number, user, df, results)
            sprinter_points(stage_number, user, df, results)
            youth_points(stage_number, user, df, results)
            itt_team_points(stage_number, user, df, results)
            flex_points(stage_number, user, df, results)
        else:
            results = create_results_dict(stage)
            gc_points(stage_number, user, df, results)
            climber_points(stage_number, user, df, results)
            sprinter_points(stage_number, user, df, results)
            youth_points(stage_number, user, df, results)
            team_points(stage_number, user, df, results)
            flex_points(stage_number, user, df, results)

    df["total_points"] = df[[f"stage_{i}" for i in range(1, 22)]].sum(axis=1)

In [ ]:
users=["Nick", "Paeyton", "Yab"]
dfs = init_dfs(users)

for user in users:
    user_score(user, dfs[user])

Processing stage 1 for Nick
Processing stage 2 for Nick
Processing stage 3 for Nick
Processing stage 4 for Nick
Processing stage 5 for Nick
Processing stage 6 for Nick
Processing stage 7 for Nick
Processing stage 8 for Nick
Processing stage 1 for Paeyton
Processing stage 2 for Paeyton
Processing stage 3 for Paeyton
Processing stage 4 for Paeyton
Processing stage 5 for Paeyton
Processing stage 6 for Paeyton
Processing stage 7 for Paeyton
Processing stage 8 for Paeyton
Processing stage 1 for Yab
Processing stage 2 for Yab
Processing stage 3 for Yab
Processing stage 4 for Yab
Processing stage 5 for Yab
Processing stage 6 for Yab
Processing stage 7 for Yab
Processing stage 8 for Yab


In [ ]:
# ---- AUTH ----
scope = ["https://spreadsheets.google.com/feeds",
         "https://www.googleapis.com/auth/spreadsheets",
         "https://www.googleapis.com/auth/drive.file",
         "https://www.googleapis.com/auth/drive"]

creds = Credentials.from_service_account_file("tour-de-frantasy-e070674b2fa6.json", scopes=scope)
client = gspread.authorize(creds)

# ---- OPEN SPREADSHEET ----
spreadsheet = client.open("Tour de Frantasy: Vuelta Edition")

# ---- LOOP THROUGH USERS ----
for user, df_user in dfs.items():
    try:
        # Try to open existing worksheet for user
        worksheet = spreadsheet.worksheet(user)
        worksheet.clear()  # clear old data
    except gspread.exceptions.WorksheetNotFound:
        # Create a new worksheet for this user
        worksheet = spreadsheet.add_worksheet(title=user, rows=str(len(df_user)+10), cols=str(len(df_user.columns)+10))

    # Write the DataFrame to the worksheet
    set_with_dataframe(worksheet, df_user)


### Next
- Create a cron job to run this automatically
- Make the spreadsheet pretty
- Speed up the code by creating a JSON with the stage results and only adding on the new results
- create a load df_lineup function so that the functions aren't calling df_lineup globally
- Add virtual/final scoring
- Add week counter and title to the scoreboard
- scrape and add combative rider / best teammate scoring